# 02 — Baselines, metrics, and persisted results

**Goal:** load the embedding cache created in notebook 01, compare all reranking baselines on identical shortlists, and save per-query metrics, summary metrics, ranked candidates, and Seaborn darkgrid figures.

The `initial_clip` control reproduces the supplied candidate ordering from `clip_score`. `encoder_similarity` uses the selected encoder's own text–image cosine. MMR, adaptive MMR, DPP, and cluster-xQuAD share that encoder score as relevance input, so we can distinguish changing the representation from changing the ranking policy.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "src" / "image_rerank").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from image_rerank.experiment import load_cached_split, evaluate_queries
from image_rerank.plots import plot_tradeoff, plot_metric_bars
from image_rerank.artifacts import ensure_dir

SPLIT = "validation"
ENCODER_KEY = "openclip-ViT-B-32-openai"  # Must match notebook 01's encoder configuration.
MAX_QUERIES = 3   # Use None once the full split has been cached.
OUTPUT_DIR = ROOT / "artifacts" / "experiments" / f"{SPLIT}_{ENCODER_KEY}_baseline"
RUN_NAME = f"{SPLIT}_{ENCODER_KEY}_baseline"

queries = load_cached_split(ROOT / "artifacts" / "cache", SPLIT, ENCODER_KEY)
if MAX_QUERIES is not None:
    queries = queries[:MAX_QUERIES]
print("Loaded cached query shortlists:", len(queries))
assert queries, "No embeddings found. Run notebook 01 first."

## 1. Run the benchmark methods

AP@50 is the main candidate-list relevance metric. nDCG@10 and Recall@10 offer top-heavy relevance views. ILD@10 is the mean pairwise cosine distance among the ten returned images: it measures visual dispersion, not whether the variety is useful. Taxonomy coverage, when available, is a descriptive secondary metric.

In [ ]:
metrics_df, summary_df = evaluate_queries(
    queries,
    top_k=50,
    diversity_k=10,
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
)

print("Per-query metrics:", metrics_df.shape)
print("Summary shape:", summary_df.shape)
display(summary_df.round(4))
display(metrics_df.head(10))

## 2. Plot the trade-off and per-metric comparisons

Both plots use the repository's `sns.set_theme(style="darkgrid")`. Treat these as exploratory until the full validation split is evaluated.

In [ ]:
figure_dir = ensure_dir(OUTPUT_DIR / "figures")
tradeoff_path = plot_tradeoff(metrics_df, figure_dir / "relevance_diversity_latency.png")
metric_path = plot_metric_bars(summary_df, figure_dir / "metric_comparison.png")
print("Saved:", tradeoff_path)
print("Saved:", metric_path)
display(Image(filename=str(tradeoff_path)))
display(Image(filename=str(metric_path)))

## 3. Inspect saved rankings

The ranking CSV has one row per query, method, and candidate rank. It records the candidate image ID, original score, encoder similarity, and binary relevance label. This makes qualitative error analysis possible without re-running the encoder.

In [ ]:
ranking_path = OUTPUT_DIR / "rankings" / f"{RUN_NAME}_rankings.csv"
rankings_df = pd.read_csv(ranking_path)
display(rankings_df.query("method == 'adaptive_mmr'").head(15))

print("Saved artifacts:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT), f"({path.stat().st_size:,} bytes)")

## Interpretation checklist

- Does AP@50 fall when ILD@10 rises?
- Is any diversity gain larger than query-to-query variation?
- Is cluster-xQuAD merely spreading across noisy clusters?
- Does a latency reduction hold across queries, or is it a few outliers?
- Are the conclusions stable when the full validation split replaces the smoke subset?

Do not select the final controller using the test split.